# Task 2: Biomaker Embedding

This task is about giving biological meaning to the biomarkers (glycans) we previously discovered by embedding them (i.e. converting each glycan into a numerical representation) into a space that reflects their biochemical, functional, and clinical relationships (i.e. such that similar glycans are placed close together in the embedding space based on these features).

In this part, we will:
- Learn a meaningful embedding space that captures relationships between glycans based on structure, origin, tissue, disease, and protein interactions.
- Validate the embedding by using the N-glycans (known structures) as a ground-truth set
- Embed our discovered glycans into this space and assess their closeness to other structures to draw conclusions as to their nature.

In [7]:
import pandas as pd
import numpy as np
import copy
import matplotlib.pyplot as plt
import time
import importlib
import os
import torch

# Machine Learning methods
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score
from tokenizers import BertWordPieceTokenizer
from transformers import (
    BertTokenizerFast,
    BertConfig,
    BertForMaskedLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)
from datasets import Dataset


# Glycowork
from glycowork.motif.processing import min_process_glycans

# Personal helpers
import helpers_TASK2
importlib.reload(helpers_TASK2)
from helpers_TASK2 import *

/Users/aishamasmoudi/opt/anaconda3/envs/isospec/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the data
glycan_list = pd.read_csv("./data/glycan_embedding/glycan_list.csv")
df_glycan = pd.read_pickle("./data/glycan_embedding/df_glycan.pkl")
glycan_binding = pd.read_pickle("./data/glycan_embedding/glycan_binding.pkl")
N_glycans_df = pd.read_pickle("./data/glycan_embedding/N_glycans_df.pkl")

# Print the shape of each dataset
print(f"Shape of Glycan List (our discovered molecules): {glycan_list.shape}")
print(f"Shape of Glycan Dataset (sequences, species, tissue, disease): {df_glycan.shape}")
print(f"Shape of Protein-Glycan binding interactions Dataset: {glycan_binding.shape}")
print(f"Shape of N-glycans Dataset (N-gylcans sequences for control representation): {N_glycans_df.shape}")

Shape of Glycan List (our discovered molecules): (5, 4)
Shape of Glycan Dataset (sequences, species, tissue, disease): (50589, 23)
Shape of Protein-Glycan binding interactions Dataset: (1465, 2745)
Shape of N-glycans Dataset (N-gylcans sequences for control representation): (48, 24)


In [3]:
# Missing Values
print(f"Glycan List:\n{glycan_list.isna().sum()}\n")
print(f"Glycan DF:\n{df_glycan.isna().sum()}\n")
print(f"Glycan Binding:\n{glycan_binding.isna().sum()}\n")
print(f"N-Glycans:\n{N_glycans_df.isna().sum()}")

Glycan List:
glycan            0
Composition       0
tissue_species    0
tissue_sample     0
dtype: int64

Glycan DF:
glycan                     0
Species                    0
Genus                      0
Family                     0
Order                      0
Class                      0
Phylum                     0
Kingdom                    0
Domain                     0
ref                        0
glytoucan_id           15573
glycan_type            23628
disease_association        0
disease_id                 0
disease_sample             0
disease_direction          0
disease_ref                0
disease_species            0
tissue_sample              0
tissue_id                  0
tissue_ref                 0
tissue_species             0
Composition                0
dtype: int64

Glycan Binding:
3-Anhydro-Gal(a1-3)Gal(b1-4)3-Anhydro-Gal(a1-3)Gal4S                                                                  1461
3-Anhydro-Gal(a1-3)Gal4S(b1-4)3-Anhydro-Gal(a1-3)Gal4S        

## Part 1: Learn a Glycan Embedding Space based on features of glycans of interest (`glycan_list`)

In this part, we will create an embedding space for the glycan libraries that captures meaningful relationships between the features of our glycans of interest, i.e. the columns of `glycan_list`:
- Sequence
- Composition
- Tissue Sample
- Tissue Species

The final goal of this part is to obtain an embedding matrix, where:
- Rows: glycans
- Columns: features (sequence, composition, tissue and species)

For this purpose, we need to find a way to represent the different features appropriately. We will investigate several ways to do this.
1. For Sequence: Using TF-IDF, Counts, or ???
2. For Composition: Using counts or ???
3. For Species: Using counts or ???
4. For Tissues: Using counts or ???

Once we obtain this representation matrix, we will apply different reducing methods to learn a meaningful embedding space. The glycans we use for the learning of the embedding space are in `df_glycans`. We will investigate several methods:
1. PCA
2. SVD
3. ???
4. ???

Finally, when the embedding space is learned, we will evaluate its quality by assessing the closeness of the N-glycans in the new embedding space.

## Part 2: Embed our glycans
Once we find a satisfying embedding space, we will embed the glycans in our `glycan_list`, and identify the *glycans* closest to our glycans of interest.

## Part 3: Enrich our glycans
The next step will be to enrich the embedding space, i.e. including more features in the representation matrix (like associated disease and protein-glycan binding interactions), and using them also to learn the embedding space. We will then identify the glycans closest to the *glycans* we found in the previous part, in order to get a distribution of information on our glycans of interest (e.g., distribution of the most likely diseases the glycan is associated to.)

## Part 1: Learn a Glycan Embedding Space based on features of glycans of interest

In part 1 of our analysis, we will build a feature space that places glycans near each other if they:
- Have similar sequences
- Have similar composition
- Come from similar species
- Come from similar tissues

To do this, we will use machine learning to learn and validate the embedding, starting with simpler, interpretable models and eventually scaling-up to more complex architectures.

In [116]:
# Load df_glycan as is done in glycowork
df_glycan2 = copy.deepcopy(df_glycan)
df_glycan2.set_index("glycan", inplace = True)
#df_glycan2.head(1).style.set_properties(**{'font-size': '8pt', 'font-family': 'Helvetica','border-collapse': 'collapse','border': '1px solid black'})

### Glycan Sequence

The first step is to find a meaningful way to represent the glycan sequences, i.e. learning an embedding space for glycan sequences, where closer glycans have similar sequences.

For this purpose, we will treat each glycan sequence as a "sentence" of glycoletters. We will try two embedding techniques:
- TF-IDF vectorizer: this converts a collection of text to a matrix of TF-IDF features. 
- Count vectorizer: this converts a collection of text to a matrix of token counts. 

Glycan composition refers to monosaccharide types and their counts. We want to create an embedding space where glycans with similar compositions (similar sugar types and counts) are close together. We will use counts of monosaccharides as the embedding.

Could also use composition_to_mass or glycan_to_mass

In [36]:
# To use later for glycan list composition embedding!!
"""
import ast

# Convert composition strings into dictionaries
glycan_list['Composition_dict'] = glycan_list['Composition'].apply(ast.literal_eval)

# Get all unique monosaccharies across the dataset
monosaccharides = set()
for composition in glycan_list['Composition_dict']:
    monosaccharides.update(composition.keys())

# Create a DataFrame with monosaccharide counts
composition_df = pd.DataFrame([
    {mono: composition.get(mono, 0) for mono in monosaccharides}
    for composition in glycan_list['Composition_dict']
])
"""

"\nimport ast\n\n# Convert composition strings into dictionaries\nglycan_list['Composition_dict'] = glycan_list['Composition'].apply(ast.literal_eval)\n\n# Get all unique monosaccharies across the dataset\nmonosaccharides = set()\nfor composition in glycan_list['Composition_dict']:\n    monosaccharides.update(composition.keys())\n\n# Create a DataFrame with monosaccharide counts\ncomposition_df = pd.DataFrame([\n    {mono: composition.get(mono, 0) for mono in monosaccharides}\n    for composition in glycan_list['Composition_dict']\n])\n"

Glycan species refers to the species of the tissue the gylcan can be found in. To create the embedding space, we one-hot encode the `tissue_species` column to obtain an embedding for each glycan. This would result in:
- Rows of embedding: glycans
- Columns: Tissue species
- Values: 1 if the glycan is associated to that tissue species, 0 if it is not

Glycan tissue refers to the tissue the gylcan can be found in. To create the embedding space, we one-hot encode the `tissue_sample` column to obtain an embedding for each glycan. This would result in:
- Rows of embedding: glycans
- Columns: Tissue
- Values: 1 if the glycan is associated to that tissue, 0 if it is not

Now, we combine all the embeddings together and normalize them.

Try with:
- TF-IDF and PCA: emb_matrix_1
- TF-IDF and SVD: emb_matrix_2
- Counts and PCA: emb_matrix_3
- Counts and SVD: emb_matrix_4

Now, we will evaluate the quality of this first embedding space. For that purpose, we will check how well the embedding space groups N-glycans, by using Silhouette Score or nearest-neighbor purity.

In [ ]:
possible_feature_list = ['TF-IDF', 'Counts', 'Composition', 'Species', 'Tissue']
possible_embedding_methods = ['PCA', 'SVD']

def learn_sequence_embedding(method, glycan_df, vocab_size=5000,
    max_length=128,
    batch_size=16,
    epochs=3,
    lr=5e-5):
    glycan_df = glycan_df.copy()
    
    # Use glycowork's min_process_glycans() to convert glycans sequence into a nested lists of glycoletters
    glycan_df['Processed Sequence'] = min_process_glycans(list(glycan_df.index))
        
    # Join glycoletters into space-separated "sentences"
    glycan_df['Sequence_str'] = glycan_df['Processed Sequence'].apply(lambda seq: ' '.join(seq))
    
    if method == 'TF-IDF':
        # Fit TF-IDF vectorizer on glycoletter sentences
        # This turns each glycan into a fixed-length vector of TF-IDF scores
        vectorier = TfidfVectorizer()
        t0 = time.time() 
        X_tfidf = vectorier.fit_transform(glycan_df['Sequence_str'])

        # Save TF-IDF glycan embeddings based on sequence similarity
        embeddings_tfidf = X_tfidf.toarray()

        # Convert TF-IDF matrix to DataFrame
        emb_seq_tfidf = pd.DataFrame(
            embeddings_tfidf,
            index=glycan_df.index,  # assumes index contains glycan identifiers
            columns=vectorier.get_feature_names_out()  # glycoletter features
            )
        
        # Time
        time_tfid = round(time.time() - t0, 2) # take into account the time it takes
        print('TF-IDF DONE')
        
        return emb_seq_tfidf, time_tfid
    
    if method=='Counts':
        # Fit Counts vectorizer on glycoletter sentences
        # This turns each gylcan into a fixed-length vector of token counts
        vec = CountVectorizer(token_pattern=r"[^ ]+")
        t0 = time.time()
        X_counts = vec.fit_transform(glycan_df['Sequence_str'])
        
        # Save Counts glycan embeddings based on sequence similarity
        embeddings_counts = X_counts.toarray()
        
        # Create DataFrame with glycoletter features
        emb_seq_counts = pd.DataFrame(
            embeddings_counts,
            index=glycan_df.index,
            columns=vec.get_feature_names_out()
            )
        
        time_counts = round(time.time() - t0, 2) # take into account the time it takes
        print('Counts DONE')
        
        return emb_seq_counts, time_counts

    if method == 'GlyBert':
        wp_tokenizer = BertWordPieceTokenizer(lowercase=False)
        wp_tokenizer.train(
            files=[seq_file],
            vocab_size=vocab_size,
            min_frequency=1,
            special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]
        )
        tokenizer = BertTokenizerFast(tokenizer_object=wp_tokenizer)
    

        

def learn_and_evaluate_embedding(feature_list, glycan_df, method, df_n_glycans):
    embeddings_list = [] # keep track of embeddings used
    total_time = 0 # keep track of time it takes to learn a specific embedding space
    
    if 'TF-IDF' in feature_list:
        emb_seq_tfidf, time_tfidf = learn_sequence_embedding('TF-IDF', glycan_df)
        embeddings_list.append(emb_seq_tfidf)
        total_time += time_tfidf
        
    if 'Counts' in feature_list:
        emb_seq_counts, time_counts = learn_sequence_embedding('Counts', glycan_df)
        embeddings_list.append(emb_seq_counts)
        total_time += time_counts
        
    """    
    if 'TF-IDF' in feature_list:
        # Fit TF-IDF vectorizer on glycoletter sentences
        # This turns each glycan into a fixed-length vector of TF-IDF scores
        vectorier = TfidfVectorizer()
        t0 = time.time() 
        X_tfidf = vectorier.fit_transform(glycan_df['Sequence_str'])

        # Save TF-IDF glycan embeddings based on sequence similarity
        embeddings_tfidf = X_tfidf.toarray()

        # Convert TF-IDF matrix to DataFrame
        emb_seq_tfidf = pd.DataFrame(
            embeddings_tfidf,
            index=glycan_df.index,  # assumes index contains glycan identifiers
            columns=vectorier.get_feature_names_out()  # glycoletter features
            )
        
        # Time
        total_time += round(time.time() - t0, 2) # take into account the time it takes

        embeddings_list.append(emb_seq_tfidf)
        print('TF-IDF DONE')
    
    if 'Counts' in feature_list:
        # Fit Counts vectorizer on glycoletter sentences
        # This turns each gylcan into a fixed-length vector of token counts
        vec = CountVectorizer(token_pattern=r"[^ ]+")
        t0 = time.time()
        X_counts = vec.fit_transform(glycan_df['Sequence_str'])
        
        # Save Counts glycan embeddings based on sequence similarity
        embeddings_counts = X_counts.toarray()
        
        # Create DataFrame with glycoletter features
        emb_seq_counts = pd.DataFrame(
            embeddings_counts,
            index=glycan_df.index,
            columns=vec.get_feature_names_out()
            )
        
        total_time += round(time.time() - t0, 2) # take into account the time it takes
        
        embeddings_list.append(emb_seq_counts)
        print('Counts DONE')
    """
        
    if 'Composition' in feature_list:
        # Get all unique monosaccharies across the dataset
        monosaccharides = set()
        t0 = time.time()
        for composition in glycan_df['Composition']:
            monosaccharides.update(composition.keys())

        # Create a DataFrame with monosaccharide counts
        emb_composition = pd.DataFrame([
            {mono: composition.get(mono, 0) for mono in monosaccharides}
            for composition in glycan_df['Composition']
        ])
        
        total_time += round(time.time() - t0, 2) # take into account the time it takes
        
        embeddings_list.append(emb_composition)
        print('Composition DONE')
    
    if 'Species' in feature_list:
        # Get the list of unique species
        seen = set() # to track seen species
        unique_species = [] # list of unique species

        t0 = time.time()
        for tissue_species in glycan_df['tissue_species']:
            for species in tissue_species:
                if species not in seen:
                    unique_species.append(species)
                    seen.add(species)
                    
        # Create a matrix of with rows = glycans, columns = tissue species, 
        # values = 1 if glycan is associated to that species, 0 if not
        emb_species = pd.DataFrame(0, index=glycan_df.index, columns=unique_species)

        # Fill matrix with 1 where species is present for each glycan
        for idx, species_list in glycan_df['tissue_species'].items():
            for species in species_list:
                if species in emb_species.columns:
                    emb_species.at[idx, species] = 1
        
        total_time += round(time.time() - t0, 2) # take into account the time it takes
        
        embeddings_list.append(emb_species)
        print('Species DONE')
        
    if 'Tissue' in feature_list:
        # Get the list of unique tissues
        seen = set() # to track seen species
        unique_tissues = [] # list of unique species

        t0 = time.time()
        for tissue_sample in df_glycan2['tissue_sample']:
            for tissue in tissue_sample:
                if tissue not in seen:
                    unique_tissues.append(tissue)
                    seen.add(tissue)
                    
        # Create a matrix of with rows = glycans, columns = tissues, 
        # values = 1 if glycan is associated to that tissue, 0 if not
        emb_tissues = pd.DataFrame(0, index=df_glycan2.index, columns=unique_tissues)

        # Fill matrix with 1 where species is present for each glycan
        for idx, tissues_list in df_glycan2['tissue_sample'].items():
            for tissue in tissues_list:
                if tissue in emb_tissues.columns:
                    emb_tissues.at[idx, tissue] = 1

        total_time += round(time.time() - t0, 2) # take into account the time it takes
                    
        embeddings_list.append(emb_tissues)
        print('Tissue DONE')
    
    if 'Disease' in feature_list:
        # Get the list of unique diseases
        seen = set() # to track seen diseases
        unique_diseases = [] # list of unique diseases

        t0 = time.time()
        for disease_association in glycan_df['disease_association']:
            for disease in disease_association:
                if disease not in seen:
                    unique_diseases.append(disease)
                    seen.add(disease)
                    
        # Create a matrix of with rows = glycans, columns = diseases, 
        # values = 1 if glycan is associated to that disease, 0 if not
        emb_diseases = pd.DataFrame(0, index=glycan_df.index, columns=unique_diseases)

        # Fill matrix with 1 where species is present for each glycan
        for idx, disease_list in glycan_df['disease_association'].items():
            for disease in disease_list:
                if disease in emb_diseases.columns:
                    emb_diseases.at[idx, disease] = 1
                    
        total_time += round(time.time() - t0, 2) # take into account the time it takes
                    
        embeddings_list.append(emb_diseases)
        print('Disease DONE')

    # Combine all embeddings together and normalize all features
    t0 = time.time()
    scaled_emb = combine_and_normalize_embeddings(embeddings_list, index=glycan_df.index) # TF-IDF
    total_time += round(time.time() - t0, 2)
    
    if method == 'PCA':
        # Learn embedding space using PCA
        t0 = time.time()
        emb_matrix = learn_embedding_PCA(scaled_emb, glycan_df.index)
        total_time += round(time.time() - t0, 2)
        print('PCA DONE')
        
    if method == 'SVD':
        # Learn embedding space using truncated SVD (i.e. Latent Semantic analysis or LSA)
        t0 = time.time()
        emb_matrix = learn_embedding_svd(scaled_emb, glycan_df.index)
        total_time += round(time.time() - t0, 2)
        print('SVD DONE')

    # Evaluate embedding method with silhouette scores
    sil_score = evaluate_embedding_sil_score(emb_matrix, df_n_glycans)
    
    # Evaluate embedding method with nearest neighbors purity
    nn_purity = evaluate_embedding_nn_purity(emb_matrix, df_n_glycans)
    
    return sil_score, nn_purity, total_time

def compare_embeddings(embedding_names, feature_lists, methods_list, df_glycan, df_n_glycans):
    """
    Evaluates multiple embedding configurations and summarizes them in a DataFrame.

    Parameters
    ----------
    feature_lists : list of list of str
        Each sublist contains the names of features used in the corresponding embedding.
        
    embedding_names : list of str
        Names (IDs) for each embedding configuration, used as row index.
        
    methods_list : list of str
        Dimensionality reduction method for each embedding (e.g. 'PCA', 'SVD').
        
    df_glycan : pd.DataFrame
        The full glycan dataset used to build embeddings.
        
    df_n_glycans : pd.DataFrame
        A DataFrame listing N-glycans for labeling and evaluation.
    
    Returns
    -------
    pd.DataFrame
        A DataFrame where:
        - Rows = embedding_names
        - Columns = all unique features (1 if present, else 0), silhouette score, NN purity, runtime (sec)
    """
    # Get all unique features
    seen = set() # to track seen features
    unique_features = [] # list of unique features

    for list in feature_lists:
        for feature in list:
            if feature not in seen:
                unique_features.append(feature)
                seen.add(feature)
    
    # Create a matrix of with rows = glycans, columns = tissues, 
    # values = 1 if glycan is associated to that tissue, 0 if not
    summary_df = pd.DataFrame(0, index=embedding_names, columns=unique_features)

    # Fill matrix with 1 where species is present for each glycan
    for name, list in zip(embedding_names, feature_lists):
        for feature in list:
            if feature in summary_df.columns:
                summary_df.at[name, feature] = 1
    
    # Prepare data to collect
    s_scores = []
    nn_purities = []
    times = []

    # Loop through embeddings
    for name, feats, method in zip(embedding_names, feature_lists, methods_list):
        print(f"Embedding name: {name}")
        print(f"Features used: {feats}")
        print(f"Method used: {method}")
        # Evaluate the embedding
        sil_score, nn_purity, time = learn_and_evaluate_embedding(feats, df_glycan, method, df_n_glycans)

        # Build feature presence dict
        s_scores.append(round(sil_score, 3))
        nn_purities.append(round(nn_purity, 3))
        times.append(round(time, 2))

    # Build DataFrame
    summary_df['Silhouette Score'] = s_scores
    summary_df['NN Purity'] = nn_purities
    summary_df['Time (s)'] = times
    summary_df.index.name = 'Embedding Name'
    
    return summary_df

IndentationError: expected an indented block (3091652106.py, line 68)

In [122]:
feature_lists = []
methods_list = []
embedding_names = []

# TF-ID Sequences, Composition, Tissue, Species using PCA
feature_lists.append(['TF-IDF', 'Composition', 'Tissue', 'Species'])
methods_list.append('PCA')
embedding_names.append('TF_IDF-Composition-Tissue-Species_PCA')

# TF-ID Sequences, Composition, Tissue, Species using SVD
"""feature_lists.append(['TF-IDF', 'Composition', 'Tissue', 'Species'])
methods_list.append('SVD')
embedding_names.append('TF_IDF-Composition-Tissue-Species_SVD')"""

# Counts Sequences, Composition, Tissue, Species using PCA
"""feature_lists.append(['Counts', 'Composition', 'Tissue', 'Species'])
methods_list.append('PCA')
embedding_names.append('Counts-Composition-Tissue-Species_PCA')"""

# Counts Sequences, Composition, Tissue, Species using SVD
feature_lists.append(['Counts', 'Composition', 'Tissue', 'Species'])
methods_list.append('SVD')
embedding_names.append('Counts-Composition-Tissue-Species_SVD')

# TF-ID Sequences, Composition, Tissue, Species, Diseases using PCA
"""feature_lists.append(['TF-IDF', 'Composition', 'Tissue', 'Species', 'Disease'])
methods_list.append('PCA')
embedding_names.append('TF_IDF-Composition-Tissue-Species_Disease_PCA')"""

summary_df1 = compare_embeddings(embedding_names, feature_lists, methods_list, df_glycan2, N_glycans_df)
summary_df1

Embedding name: TF_IDF-Composition-Tissue-Species_PCA
Features used: ['TF-IDF', 'Composition', 'Tissue', 'Species']
Method used: PCA
TF-IDF DONE
Composition DONE
Species DONE
Tissue DONE
Shape of combined embeddings before reduction: (50589, 2553)
Shape of final Embedding Matrix after PCA: (50589, 50)
PCA DONE
Silhouette Score: 0.407
Nearest-Neighbor Purity (k=5): 0.117
Embedding name: Counts-Composition-Tissue-Species_SVD
Features used: ['Counts', 'Composition', 'Tissue', 'Species']
Method used: SVD
Counts DONE
Composition DONE
Species DONE
Tissue DONE
Shape of combined embeddings before reduction: (50589, 2894)
Shape of final Embedding Matrix after SVD: (50589, 50)
SVD DONE


KeyboardInterrupt: 

TF-IDF with PCA:
- Silhouette Score: 0.406 --> moderate to strong score. N-glycans are fairly well separated from non-N-glycans in our embedding space.
- NN Purity : 0.108 --> low score. On average, only ~11% of an N-glycan's neighbors are also N-glycans. N-glycans are not grouped together locally. It is possible that either the embedding space is not the best, or the N-glycans are spread across multiple regions, or the N-glycans are outnumbered in the data (imbalance).

--> This embedding is doing an okay job of separating glycan types overall, but not tightly grouping similar ones together.

TF-IDF with SVD:
- Silhouette Score: 0.403
- NN Purity: 0.113

--> Similar results to TF-IDF with PCA

Counts with PCA:
- Silhouette Score: 0.358
- NN Purity: 0.088

Counts with SVD:
- Silhouette Score: 0.254
- NN purity: 0.079

---
Now that we have combined the features: sequence proximity, composition, tissue and species, we will add this following feature:
- Disease Association

---
Now that we have combined the features: sequence proximity, composition, tissue, species, and disease assocation, we will add this following feature: 
- Protein-Glycan Binding

---

In [100]:
glycan_binding['3-Anhydro-Gal(a1-3)Gal(b1-4)3-Anhydro-Gal(a1-3)Gal4S'].dropna()

237     0.010219
273    -0.423532
533    -0.310640
1379   -0.075725
Name: 3-Anhydro-Gal(a1-3)Gal(b1-4)3-Anhydro-Gal(a1-3)Gal4S, dtype: float64

In [101]:
glycan_binding.shape

(1465, 2745)

In [111]:
glycan_binding.head()

,3-Anhydro-Gal(a1-3)Gal(b1-4)3-Anhydro-Gal(a1-3)Gal4S,3-Anhydro-Gal(a1-3)Gal4S(b1-4)3-Anhydro-Gal(a1-3)Gal4S,3-Anhydro-Gal(a1-3)Gal4S(b1-4)3-Anhydro-Gal(a1-3)Gal4S(b1-4)3-Anhydro-Gal(a1-3)Gal4S,3-Anhydro-Gal(a1-3)Gal4S(b1-4)3-Anhydro-Gal(a1-3)Gal4S(b1-4)3-Anhydro-Gal(a1-3)Gal4S(b1-4)3-Anhydro-Gal(a1-3)Gal4S,3-Anhydro-Gal(a1-3)Gal4S(b1-4)3-Anhydro-Gal2S(a1-3)Gal4S(b1-4)3-Anhydro-Gal(a1-3)Gal4S,3dGal(b1-3)[Fuc(a1-4)]Glc,3dGal(b1-4)Glc,4d8dNeu5Ac(a2-3)Gal(b1-4)Glc,4dNeu5Ac(a2-3)Gal(b1-4)Glc,7dNeu5Ac(a2-3)Gal(b1-4)Glc,...,wwwSflexneri5c,wwwSflexneriO2c,wwwSflexneriO5c,wwwSisomicin,wwwSmix,wwwTobramycin,wwwTyrS,wwwpHGGs,target,protein
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AADSIPSISPTGIITPTPTQSGMVSNCNKFYDVHSNDGCSAIASSQ...,TAL6-4LysM
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AAFFSLVVLLALLPFGIHASALPSTELTPRVNPNLPGPNDVFVGFR...,rCnSL-proA
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AANEADYQAKLTAYQTELARVQKANADAKAAYEAAVAANNAANAAL...,AntigenI/IIA3VP1
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AASKLGVPQPAQRDQVNCQLYAVQPNDNCIDISSKNNITYAQLLSW...,TAL6-6LysM
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ACNNEWEDEQYEQYISFKSPIPAGGEGVTDIYVRYKEDGKVTYRLP...,SP15308A-bot-339-19-339


So we:
- Learn embedding space based on sequences, composition, tissue and species.
- Represent glycans from glycan_list in this space.
- Identify close glycans to each glycan in this space.
- Enrich the glycans with context like: diseases associated with nearby glycans and known protein-binding partners.

In [112]:
glycan_list

,glycan,Composition,tissue_species,tissue_sample
0,Fuc(a1-?)GlcNAc(b1-2)Man(a1-6)[GlcNAc(b1-2)Man...,"{'dHex': 2, 'HexNAc': 4, 'Hex': 3}",['Homo_sapiens'],['blood']
1,Neu5Ac(a2-?)Gal(b1-4)GlcNAc(b1-2)Man(a1-6)[Glc...,"{'Neu5Ac': 1, 'Hex': 4, 'HexNAc': 4, 'dHex': 1}",['Homo_sapiens'],['blood']
2,Neu5Ac(a2-6)Gal(b1-4)GlcNAc(b1-2)Man(a1-6)[Gal...,"{'Neu5Ac': 1, 'Hex': 5, 'HexNAc': 4}",['Homo_sapiens'],['blood']
3,Neu5Ac(a2-6)Gal(b1-4)GlcNAc(b1-2)Man(a1-6)[Glc...,"{'Neu5Ac': 1, 'Hex': 4, 'HexNAc': 4}",['Homo_sapiens'],['blood']
4,Fuc(a1-2)[GalNAc(a1-3)]Gal(b1-4)GlcNAc(b1-2)Ma...,"{'dHex': 1, 'HexNAc': 5, 'Hex': 5}",['Homo_sapiens'],['blood']


# TODO
- Try BERT for sequence embedding
- Try other embeddings for composition, tissues, species
- Embed glycans in this space
etc.